# Phase 3 — TRL SFT on the SPY + Nifty 50 OpenEnv (Hackathon)

This is the **training notebook** for the submission adapter
[`Kj2461/metaOpenNV-sft-qwen15`](https://huggingface.co/Kj2461/metaOpenNV-sft-qwen15).

**Base model:** `Qwen/Qwen2.5-1.5B-Instruct` + LoRA (r=16, alpha=32, q/k/v/o + MLP).
**Datasets:** SPY (US) and Nifty 50 / RELIANCE.NS (IN) — the OpenEnv alternates between them on every `reset()`, so a single 30-episode collection produces a balanced 50/50 mix (15 SPY + 15 Nifty 50, 11,700 SFT rows).

**Runtime:** enable a **GPU** (T4 or better).
**Secrets (Colab):** add `HF_TOKEN` (Hub write scope, only needed if you push the trained adapter back to the Hub).

**Flow:**
1. Clone repo → install TRL stack
2. (Already committed) Mixed JSONL `data/trl_sft_train_mixed.jsonl` — 11,700 rows from both markets
3. Run `SFTTrainer` → view loss plot
4. Optional: head-to-head eval base vs SFT on the live OpenEnv

**Production training (the version on the Hub) was run on a Hugging Face GPU Space (`Kj2461/metaOpenNV_V2-train`) using `Dockerfile.train` + `scripts/hf_train_and_push.py` — same code, same data, same hyperparameters as the cells below.** This notebook is the reproducible Colab fallback.

In [ ]:
import os
import subprocess

os.environ["PYTHONUTF8"] = "1"

REPO_URL = "https://github.com/kunaljaiswal2461-lab/metaOpenNV_V2.git"
DIR = "metaOpenNV_V2"
if not os.path.isdir(DIR):
    subprocess.check_call(["git", "clone", REPO_URL])
os.chdir(DIR)

In [ ]:
!pip install -q -r requirements-trl.txt

In [ ]:
import os
from google.colab import userdata

# Live Space that exposes /reset and /step
os.environ["SPACE_URL"] = "https://huggingface.co/spaces/Kj2461/metaOpenNV_V2"

try:
    tok = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = tok
except Exception:
    print("Set HF_TOKEN in Colab secrets if you need authenticated Hub access.")

In [ ]:
# The repo ships a pre-collected, balanced SPY + Nifty 50 JSONL at
# data/trl_sft_train_mixed.jsonl (11,700 rows, 15 SPY + 15 Nifty 50 episodes).
# Skip directly to training.
#
# To regenerate locally instead, uncomment:
# !python scripts/collect_sft_dataset.py --local --episodes 30 --max-steps 400 \
#     --task-name multi_horizon_trading --prompt compact --teacher sma20 \
#     --out data/trl_sft_train_mixed.jsonl
# (Expected output tail: "Episode dataset mix: nifty50=15, spy=15  Wrote 11700 rows")
!head -1 data/trl_sft_train_mixed.jsonl
!wc -l data/trl_sft_train_mixed.jsonl

In [ ]:
# TRL SFT on Qwen 2.5 1.5B Instruct with LoRA, 1 epoch on the mixed JSONL.
# This reproduces the production adapter at https://huggingface.co/Kj2461/metaOpenNV-sft-qwen15
# Wall clock on T4 medium: ~1 hour for 1,170 optimisation steps (acc=4).
!python scripts/trl_sft_train.py \
    --data data/trl_sft_train_mixed.jsonl \
    --model-id Qwen/Qwen2.5-1.5B-Instruct \
    --epochs 1 \
    --max-length 512 \
    --output-dir results/phase3_lora

In [ ]:
from IPython.display import Image, display
display(Image("results/trl_sft_loss.png"))

In [ ]:
# Optional: head-to-head eval on the live OpenEnv (writes results/phase3_eval_*).
# Compares the un-tuned 1.5B base vs the LoRA we just trained vs the production
# adapter on the Hub. The env alternates SPY <-> Nifty 50 so this is a fair
# cross-market test.
!python scripts/eval_llm_on_env.py \
    --models base=Qwen/Qwen2.5-1.5B-Instruct \
             sft_local=results/phase3_lora \
             sft_hub=Kj2461/metaOpenNV-sft-qwen15 \
    --episodes 10 --max-steps 300 --task-name multi_horizon_trading --local

from IPython.display import Image, display
display(Image("results/phase3_eval_bar.png"))